In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.ensemble import IsolationForest
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ------------------------------
# 1. Загрузка данных и сортировка
# ------------------------------
train = pd.read_csv('magline_train_df.csv')
test = pd.read_csv('magline_test_df.csv')

test_ids = test['id'].copy()

# Сортировка по id (предполагаем временной порядок)
train = train.sort_values('id').reset_index(drop=True)
test = test.sort_values('id').reset_index(drop=True)

# Проверка наличия обязательных колонок
required_cols = ['s', 'h', 'v']
for col in required_cols:
    if col not in train.columns:
        raise ValueError(f"Column {col} not found in train data")

# ------------------------------
# 2. Обработка пропусков в сигналах
# ------------------------------
signal_cols = ['m_minus5', 'm_minus4', 'm_minus3', 'm_minus2', 'm_minus1',
               'm_0', 'm_plus1', 'm_plus2', 'm_plus3', 'm_plus4', 'm_plus5']

# Флаги пропусков
for col in signal_cols:
    train[col + '_missing'] = train[col].isna().astype(int)
    test[col + '_missing'] = test[col].isna().astype(int)

# Интерполяция (линейная по порядку строк)
train[signal_cols] = train[signal_cols].interpolate(method='linear', limit_direction='both')
test[signal_cols] = test[signal_cols].interpolate(method='linear', limit_direction='both')

# Заполнение оставшихся пропусков медианой
for col in signal_cols:
    train[col].fillna(train[col].median(), inplace=True)
    test[col].fillna(test[col].median(), inplace=True)

# ------------------------------
# 3. Признаки на основе сигнала в окне
# ------------------------------
def add_signal_features(df):
    # Статистики окна
    df['m_mean'] = df[signal_cols].mean(axis=1)
    df['m_median'] = df[signal_cols].median(axis=1)
    df['m_std'] = df[signal_cols].std(axis=1)
    df['m_min'] = df[signal_cols].min(axis=1)
    df['m_max'] = df[signal_cols].max(axis=1)
    df['m_range'] = df['m_max'] - df['m_min']
    df['m_skew'] = df[signal_cols].skew(axis=1)
    df['m_kurt'] = df[signal_cols].kurtosis(axis=1)
    df['m_cv'] = df['m_std'] / (df['m_mean'].abs() + 1e-9)

    # Отклонения центрального значения
    df['m0_diff_mean'] = df['m_0'] - df['m_mean']
    df['m0_diff_median'] = df['m_0'] - df['m_median']

    # Градиенты (последовательные разности) – создаём, потом удалим, оставив агрегаты
    grad_cols = []
    for i in range(len(signal_cols)-1):
        grad = f'grad_{i}'
        df[grad] = df[signal_cols[i+1]] - df[signal_cols[i]]
        grad_cols.append(grad)

    # Агрегаты градиентов
    df['grad_sum_abs'] = df[grad_cols].abs().sum(axis=1)
    df['grad_max_abs'] = df[grad_cols].abs().max(axis=1)
    df['grad_mean_abs'] = df[grad_cols].abs().mean(axis=1)

    # Вторые разности
    grad2_cols = []
    for i in range(len(grad_cols)-1):
        grad2 = f'grad2_{i}'
        df[grad2] = df[grad_cols[i+1]] - df[grad_cols[i]]
        grad2_cols.append(grad2)
    df['grad2_sum_abs'] = df[grad2_cols].abs().sum(axis=1)

    # Удаляем промежуточные градиенты
    df.drop(columns=grad_cols + grad2_cols, inplace=True, errors='ignore')

    # Локальные экстремумы
    left = df[signal_cols[:5]].values
    right = df[signal_cols[6:]].values
    m0 = df['m_0'].values[:, np.newaxis]
    df['is_local_max'] = ((m0 > left).all(axis=1) & (m0 > right).all(axis=1)).astype(int)
    df['is_local_min'] = ((m0 < left).all(axis=1) & (m0 < right).all(axis=1)).astype(int)

    return df

train = add_signal_features(train)
test = add_signal_features(test)

# ------------------------------
# 4. Скользящие признаки по времени для сигналов и их агрегатов
# ------------------------------
def add_rolling_signal_features(df, windows=[3, 5, 10]):
    # Для исходных сигналов
    for col in signal_cols:
        for w in windows:
            df[f'{col}_roll_mean_{w}'] = df[col].rolling(w, min_periods=1).mean()
            df[f'{col}_roll_std_{w}'] = df[col].rolling(w, min_periods=1).std().fillna(0)
    # Для агрегатов (если они есть)
    for agg in ['m_mean', 'm_std', 'm_median']:
        if agg in df.columns:
            for w in windows:
                df[f'{agg}_roll_mean_{w}'] = df[agg].rolling(w, min_periods=1).mean()
                df[f'{agg}_roll_std_{w}'] = df[agg].rolling(w, min_periods=1).std().fillna(0)
    return df

train = add_rolling_signal_features(train)
test = add_rolling_signal_features(test)

# ------------------------------
# 5. Признаки пути, высоты, скорости
# ------------------------------
def add_derivatives(df):
    for col in ['s', 'h', 'v']:
        df[f'{col}_diff'] = df[col].diff().fillna(0)
        df[f'{col}_diff2'] = df[f'{col}_diff'].diff().fillna(0)
    for col in ['h', 'v']:
        df[f'{col}_rolling_mean_5'] = df[col].rolling(5, min_periods=1).mean()
        df[f'{col}_rolling_std_5'] = df[col].rolling(5, min_periods=1).std().fillna(0)
    return df

train = add_derivatives(train)
test = add_derivatives(test)

# Взаимодействия
train['h_v'] = train['h'] * train['v']
train['s_h'] = train['s'] * train['h']
test['h_v'] = test['h'] * test['v']
test['s_h'] = test['s'] * test['h']

# ------------------------------
# 6. Isolation Forest (unsupervised)
# ------------------------------
# Базовый набор признаков (без Isolation Forest и без целевой)
base_feature_cols = [col for col in train.columns if col not in ['id', 'is_anomaly'] + signal_cols]
print(f'Базовое количество признаков: {len(base_feature_cols)}')

X_base = train[base_feature_cols].values

# Обучаем Isolation Forest (contamination можно подобрать по доле аномалий в train)
contamination = 0.05  # предположительная доля аномалий
iso_forest = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)
iso_forest.fit(X_base)

# Добавляем признаки в train
train['iso_anomaly_score'] = iso_forest.decision_function(X_base)
train['iso_pred'] = iso_forest.predict(X_base)

# Для теста
X_test_base = test[base_feature_cols].values
test['iso_anomaly_score'] = iso_forest.decision_function(X_test_base)
test['iso_pred'] = iso_forest.predict(X_test_base)

# Итоговый список признаков (включая Isolation Forest)
feature_cols = base_feature_cols + ['iso_anomaly_score', 'iso_pred']
print(f'Итоговое количество признаков: {len(feature_cols)}')

X = train[feature_cols].values
y = train['is_anomaly'].values

X_test = test[feature_cols].values

# ------------------------------
# 7. Разделение на train / valid с сохранением временного порядка
# ------------------------------
split_idx = int(0.8 * len(X))
X_train, X_valid = X[:split_idx], X[split_idx:]
y_train, y_valid = y[:split_idx], y[split_idx:]

print(f'Train size: {len(X_train)}, Valid size: {len(X_valid)}')
print('Train class distribution:\n', pd.Series(y_train).value_counts())
print('Valid class distribution:\n', pd.Series(y_valid).value_counts())

# ------------------------------
# 8. Обучение LightGBM с учётом дисбаланса
# ------------------------------
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model.fit(X_train, y_train,
          eval_set=[(X_valid, y_valid)],
          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

# ------------------------------
# 9. Подбор порога по F1 на валидации
# ------------------------------
y_pred_proba = model.predict_proba(X_valid)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_valid, y_pred_proba)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f'Best threshold: {best_threshold:.3f}, F1 on valid: {best_f1:.4f}')

# ------------------------------
# 10. Предсказание на тесте
# ------------------------------
test_pred_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_pred_proba >= best_threshold).astype(int)

# ------------------------------
# 11. Сохранение результата
# ------------------------------
submission = pd.DataFrame({'id': test_ids, 'isanomaly': test_pred})
submission.to_csv('submit_lgbm_rolling.csv', index=False)
print('Submission saved to submit_lgbm_rolling.csv')

Базовое количество признаков: 127
Итоговое количество признаков: 129
Train size: 75897, Valid size: 18975
Train class distribution:
 0    68197
1     7700
Name: count, dtype: int64
Valid class distribution:
 0    17417
1     1558
Name: count, dtype: int64
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.109295
[200]	valid_0's binary_logloss: 0.0927676
[300]	valid_0's binary_logloss: 0.0852266
[400]	valid_0's binary_logloss: 0.0807163
[500]	valid_0's binary_logloss: 0.0789584
Did not meet early stopping. Best iteration is:
[500]	valid_0's binary_logloss: 0.0789584
Best threshold: 0.731, F1 on valid: 0.8324
Submission saved to submit_lgbm_rolling.csv
